## **GENERACIÓN DE FIGURAS PARA EL CAPÍTULO DE RESULTADOS DE LA MEMORIA**

In [2]:
import os
import matplotlib.pyplot as plt
import numpy as np

OUTPUT_DIR = "figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "figure.dpi": 150,
})


In [3]:
def plot_quality_vs_latency():
    """Trade-off calidad vs latencia en Kvasir-SEG."""
    data = {
        "U-Net":         (8.9,   0.828, "classical"),
        "YOLOv8n-Seg":   (13.5,  0.832, "classical"),
        "SAM Base":      (102.2, 0.786, "sam1"),
        "SAM Large":     (210.7, 0.756, "sam1"),
        "SAM 2 Base":    (79.7,  0.850, "sam2"),
        "SAM 2 Large":   (140.4, 0.872, "sam2"),
        "SAM 2.1 Base":  (79.5,  0.858, "sam2_1"),
        "SAM 2.1 Large": (140.9, 0.861, "sam2_1"),
        "SAM 3":         (424.0, 0.878, "sam3"),
    }

    colors = {
        "classical": "#2C7BB6",
        "sam1":      "#ABD9E9",
        "sam2":      "#FDAE61",
        "sam2_1":    "#F46D43",
        "sam3":      "#D7191C",
    }
    labels = {
        "classical": "Arquitecturas clásicas",
        "sam1":      "SAM 1",
        "sam2":      "SAM 2",
        "sam2_1":    "SAM 2.1",
        "sam3":      "SAM 3",
    }

    fig, ax = plt.subplots(figsize=(8, 5.2))

    plotted_families = set()
    for name, (lat, miou, family) in data.items():
        label = labels[family] if family not in plotted_families else None
        ax.scatter(lat, miou, s=120, color=colors[family],
                   edgecolor="black", linewidth=0.7, label=label, zorder=3)
        plotted_families.add(family)

        offset_x = 1.08 if "Large" not in name else 1.08
        ax.annotate(name, (lat, miou), xytext=(lat * offset_x, miou + 0.005),
                    fontsize=8, ha="left", va="bottom")

    ax.axvline(40, color="gray", linestyle="--", linewidth=1, alpha=0.6)
    ax.text(41, 0.760, "Umbral tiempo real (40 ms)",
            fontsize=8, color="gray", rotation=90, va="bottom")

    ax.set_xscale("log")
    ax.set_xlabel("Latencia por imagen (ms, escala logarítmica)")
    ax.set_ylabel("mIoU")
    ax.set_title("Calidad de segmentación frente a coste computacional en Kvasir-SEG")
    ax.grid(True, which="both", linestyle=":", alpha=0.4)
    ax.legend(loc="lower right", framealpha=0.95)
    ax.set_ylim(0.74, 0.90)

    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, "tradeoff_calidad_latencia.pdf")
    plt.savefig(out, bbox_inches="tight")
    plt.close()
    print(f"Generado: {out}")


In [4]:
def plot_evolution_zero_shot():
    """Evolución del mIoU zero-shot por generación de SAM en cada dataset."""
    models = ["SAM\nBase", "SAM\nLarge", "SAM 2\nBase", "SAM 2\nLarge",
              "SAM 2.1\nBase", "SAM 2.1\nLarge", "SAM 3"]

    datasets = {
        "Kvasir-SEG":       [0.577, 0.637, 0.625, 0.643, 0.637, 0.666, 0.779],
        "PASCAL-S":         [0.363, 0.423, 0.609, 0.619, 0.617, 0.615, 0.670],
        "RefCOCOg":         [0.776, 0.821, 0.841, 0.840, 0.840, 0.842, 0.843],
        "ISIC 2016":        [0.566, 0.582, 0.613, 0.629, 0.605, 0.621, 0.755],
        "Mapillary Vistas": [0.358, 0.367, 0.392, 0.376, 0.401, 0.389, 0.403],
    }

    colors = ["#1B9E77", "#D95F02", "#7570B3", "#E7298A", "#66A61E"]
    markers = ["o", "s", "^", "D", "v"]

    fig, ax = plt.subplots(figsize=(9, 5.2))

    x = np.arange(len(models))
    for (name, values), color, marker in zip(datasets.items(), colors, markers):
        ax.plot(x, values, marker=marker, linewidth=1.8, markersize=7,
                color=color, label=name, markeredgecolor="black",
                markeredgewidth=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels(models)
    ax.set_ylabel("mIoU")
    ax.set_title("Evolución del rendimiento $\\it{zero\\text{-}shot}$ por generación de SAM")
    ax.grid(True, axis="y", linestyle=":", alpha=0.5)
    ax.legend(loc="lower right", ncol=2, framealpha=0.95)
    ax.set_ylim(0.30, 0.90)

    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, "evolucion_zero_shot.pdf")
    plt.savefig(out, bbox_inches="tight")
    plt.close()
    print(f"Generado: {out}")


In [5]:
def plot_zs_vs_ft():
    """Comparativa zero-shot vs fine-tuning por dataset (excepto Mapillary)."""
    models = ["SAM\nBase", "SAM\nLarge", "SAM 2\nBase", "SAM 2\nLarge",
              "SAM 2.1\nBase", "SAM 2.1\nLarge", "SAM 3"]

    datasets = {
        "Kvasir-SEG": {
            "ZS": [0.577, 0.637, 0.625, 0.643, 0.637, 0.666, 0.779],
            "FT": [0.786, 0.756, 0.850, 0.872, 0.858, 0.861, 0.878],
        },
        "PASCAL-S": {
            "ZS": [0.363, 0.423, 0.609, 0.619, 0.617, 0.615, 0.670],
            "FT": [0.800, 0.828, 0.852, 0.876, 0.845, 0.858, 0.891],
        },
        "ISIC 2016": {
            "ZS": [0.566, 0.582, 0.613, 0.629, 0.605, 0.621, 0.755],
            "FT": [0.753, 0.713, 0.831, 0.837, 0.828, 0.830, 0.854],
        },
        "RefCOCOg": {
            "ZS": [0.776, 0.821, 0.841, 0.840, 0.840, 0.842, 0.843],
            "FT": [0.398, 0.789, 0.790, 0.805, 0.774, 0.812, 0.839],
        },
    }

    fig, axes = plt.subplots(2, 2, figsize=(11, 7.5), sharey=True)
    axes = axes.flatten()

    x = np.arange(len(models))
    width = 0.38

    for ax, (name, values) in zip(axes, datasets.items()):
        ax.bar(x - width / 2, values["ZS"], width, label="Zero-shot",
               color="#9ECAE1", edgecolor="black", linewidth=0.6)
        ax.bar(x + width / 2, values["FT"], width, label="Fine tuning",
               color="#3182BD", edgecolor="black", linewidth=0.6)

        ax.set_xticks(x)
        ax.set_xticklabels(models, fontsize=8)
        ax.set_title(name)
        ax.grid(True, axis="y", linestyle=":", alpha=0.4)
        ax.set_ylim(0, 1.0)

    for ax in axes[::2]:
        ax.set_ylabel("mIoU")

    axes[0].legend(loc="lower right", framealpha=0.95)

    fig.suptitle("Comparativa del rendimiento $\\it{zero\\text{-}shot}$ frente al $\\it{fine\\ tuning}$",
                 fontsize=12, y=1.00)

    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, "comparativa_zs_vs_ft.pdf")
    plt.savefig(out, bbox_inches="tight")
    plt.close()
    print(f"Generado: {out}")


In [6]:
if __name__ == "__main__":
    plot_quality_vs_latency()
    plot_evolution_zero_shot()
    plot_zs_vs_ft()

Generado: figures\tradeoff_calidad_latencia.pdf
Generado: figures\evolucion_zero_shot.pdf
Generado: figures\comparativa_zs_vs_ft.pdf
